# Inference Masterclass I: Quantization/Harness-En

**Repository author and maintainer:** [Angel Galvis](https://github.com/angelgalvisc) · [LinkedIn](https://www.linkedin.com/in/angelgalvisc/)

## Abstract

Inference services provided by frontier AI labs have accelerated enterprise adoption and made extraordinary capabilities accessible through an API. Under this consumption model, however, the provider retains control over the model weights, runtime, compute allocation, and model evolution. For organizations operating high-volume inference workloads, open-weight models offer a strategic alternative: intelligence can be specialized, quantized, and deployed on in-house or dedicated infrastructure sized for each workload. The model, its operating cost, and its evolution therefore become architectural decisions the organization can control.

This notebook explores that opportunity through Alchemist, a 4-billion-parameter agentic model quantized to 4 bits and reduced to approximately 2.4 GB. The demonstration covers its installation and execution on a Kaggle GPU, compares its behavior with the full-precision model, and examines the extent to which quantization preserves its semantic and task-solving capabilities.

Based on the observed behavior, the notebook constructs a minimalist harness that introduces output contracts, execution limits, independent validation, bounded recovery, and deterministic operations. The model interprets the request and proposes a structured action; the validator checks that the action satisfies the contract; and the executor applies the rules and calculations that require exactness. Preserved traces make every transition observable and allow the reader to reconstruct what the model proposed, what the system accepted, and which operation was ultimately executed.

The result is a hybrid architecture: frontier models remain available for open-ended problems, while specialized open-weight models can serve repeatable, private, and high-volume workloads. This shift toward **self-hosted intelligence** is not simply about running a smaller model; it is about turning that model into an auditable and adaptable unit of intelligence that can be deployed on resources the organization can size and govern.

Welcome! In this notebook, you will install and run [Agent A1 Alchemist 4-bit](https://huggingface.co/angelgalvisc/agent-a1-alchemist-4bit) on a Kaggle GPU for the first time.

## What is Alchemist?

Alchemist is a compact, quantized, MLX-ready release of a four-billion-parameter agentic model. Its lineage is:

1. [Qwen3.5-4B](https://huggingface.co/Qwen/Qwen3.5-4B), developed by Alibaba's Qwen team, provides the foundation language model.
2. [Agents-A1-4B](https://huggingface.co/InternScience/Agents-A1-4B), developed by InternScience, builds on Qwen3.5-4B with agentic training over tool-use trajectories.
3. [Agent A1 Alchemist 4-bit](https://huggingface.co/angelgalvisc/agent-a1-alchemist-4bit) was quantized and packaged by **Datastrat** from Agents-A1-4B.

### How did Datastrat quantize the model?

**Datastrat developed Alchemist as a compact MLX release of Agents-A1-4B using a mixed-precision quantization strategy designed to preserve agentic behavior.** The process can be summarized as five technical decisions:

1. **Group-wise affine quantization**

   The language module's 248 principal weight matrices were divided into groups of 128 weights. Each group is represented with 4-bit values and retains its own scale and zero point. This adapts the numerical representation to each group's local range instead of applying a single scale to an entire matrix.

2. **Per-channel activation rescaling**

   Before rounding, Datastrat rescaled 216 of the 248 matrices. Weights connected to higher-magnitude activation channels were amplified so that 4-bit rounding would preserve their information more accurately.

3. **Scale folding into neighboring layers**

   The preceding rescaling was compensated by dividing the corresponding activations by the same factor. That inverse operation was folded algebraically into a neighboring layer, preserving the model's computation before rounding without adding inference-time operations or increasing the file size. In practice, this reduces quantization error in the most sensitive channels.

4. **Mixed-precision allocation**

   Not every component was compressed in the same way:

   - The 248 principal projections were stored nominally at 4 bits.
   - The shared input/output vocabulary table was retained at 6 bits because evaluation revealed a sharp degradation below that precision.
   - Normalization parameters remained in BF16.
   - The 32 attention-output matrices were quantized without rescaling because there is no suitable preceding linear layer into which the compensating division can be folded.

5. **Full-range preservation (no range clipping)**

   Datastrat evaluated clipping extreme weight values, a technique that can improve some conventional text metrics. However, the evaluations also showed that clipping increased repeated failed tool calls. Alchemist therefore preserves the full weight range and prioritizes the stability of agentic behavior.

Although Alchemist is described as a “4-bit model,” that label is a useful shorthand:

- The projections require approximately 4.25 bits per weight once scales and zero points are included.
- The vocabulary table requires approximately 6.25 bits per weight.
- The language module averages **4.555 bits per weight**.
- The resulting weight file occupies approximately **2.395 GB**.

The result is [Agent A1 Alchemist 4-bit](https://huggingface.co/angelgalvisc/agent-a1-alchemist-4bit), an MLX checkpoint derived from [Agents-A1-4B](https://huggingface.co/InternScience/Agents-A1-4B) that substantially reduces storage and memory requirements while seeking to preserve the parent model's agentic capabilities.

In the following sections, we will configure MLX's CUDA backend, confirm that Kaggle assigned a compatible NVIDIA GPU, load the model directly from Hugging Face, and generate its first response. This working installation provides the foundation for using Alchemist with the Alchemist-RLM Harness.


## 1. Configure the Kaggle session

Before running the notebook, open **Settings → Accelerator** and select **GPU T4 ×2**. Keep **Internet on** so the model can be downloaded from Hugging Face.

> Kaggle may expose two T4 GPUs, while this introductory single-model inference uses the default GPU.

## 2. Verify the GPU and install MLX for CUDA

`mlx-lm` provides model loading and text generation. On Linux with an NVIDIA GPU, MLX also needs its CUDA backend. Kaggle's current T4 environment uses a CUDA 13-compatible driver, so both packages are pinned below to the versions used in this masterclass.

In [1]:
# Display the NVIDIA GPUs assigned to this Kaggle session.
!nvidia-smi

# Install reproducible versions of MLX with CUDA support and MLX-LM.
%pip install -q --upgrade "mlx[cuda13]==0.32.1" "mlx-lm==0.31.3"

Wed Aug 19 09:56:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 3. Confirm that MLX is using the GPU

The following cell records the installed versions and verifies that MLX selected a CUDA device. You should see `CUDA available: True` and a device such as `Device(gpu, 0)`.

In [2]:
from importlib.metadata import version
import mlx.core as mx
from mlx_lm import load, generate

print("MLX version:", version("mlx"))
print("MLX-LM version:", version("mlx-lm"))
print("CUDA available:", mx.cuda.is_available())
print("Active device:", mx.default_device())

MLX version: 0.32.1
MLX-LM version: 0.31.3
CUDA available: True
Active device: Device(gpu, 0)


## 4. Load Alchemist

MLX-LM downloads the checkpoint from its [Hugging Face repository](https://huggingface.co/angelgalvisc/agent-a1-alchemist-4bit) and initializes the tokenizer and model.

The checkpoint weights remain quantized to 4 bits. Because Tesla T4 GPUs are optimized for FP16 and do not provide native BF16 tensor-core execution, `set_dtype(mx.float16)` selects FP16 for the model's floating-point parameters and operations. This is the compatible execution mode used for this Kaggle setup.

In [3]:
# Download and load Alchemist directly from Hugging Face.
model, tokenizer = load("angelgalvisc/agent-a1-alchemist-4bit")

# Keep the checkpoint quantized while using T4-compatible FP16 operations.
model.set_dtype(mx.float16)

print("Alchemist is loaded and ready.")

You are using a model of type qwen3_5 to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


Alchemist is loaded and ready.


## 5. Generate the first response

We format the conversation with Alchemist's own chat template and disable the optional thinking trace for a short introductory response. `verbose=True` also reports prompt speed, generation speed, and peak GPU memory.

In [4]:
messages = [
    {"role": "user", "content": "Say hello briefly and introduce yourself."}
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

response = generate(
    model,
    tokenizer,
    prompt=prompt,
    max_tokens=96,
    verbose=True,
)

print("\nAlchemist response:")
print(response)

Hello! I'm Agent-A1, a deep research assistant built by Datastrat. I'm here to help you with any questions or tasks you have. How can I assist you today?
Prompt: 378 tokens, 4.399 tokens-per-sec
Generation: 40 tokens, 34.140 tokens-per-sec
Peak memory: 3.579 GB

Alchemist response:
Hello! I'm Agent-A1, a deep research assistant built by Datastrat. I'm here to help you with any questions or tasks you have. How can I assist you today?


## 6. A second guided question: understanding quantization

This second interaction teaches the idea behind the checkpoint Alchemist is running. The question is deliberately simple: what quantization is, why a 4-bit model can use less memory than a 16-bit or 32-bit one, and what the main drawback of lower precision can be. Asking for a single short paragraph keeps the answer readable and easy to compare with your own run.

In [5]:
quantization_question = (
    "In simple English, what is model quantization? "
    "Why can a 4-bit model use less memory than a 16-bit or 32-bit model, "
    "and what trade-off can lower precision introduce? "
    "Answer in one short paragraph."
)

prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": quantization_question}],
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

quantization_answer = generate(
    model,
    tokenizer,
    prompt=prompt,
    max_tokens=384,
    verbose=True,
)

print("\nAlchemist's explanation:")
print(quantization_answer)

Model quantization is a technique that reduces the number of bits used to represent each numerical value in a machine learning model, typically converting high-precision weights (like 32-bit floats) into lower precision formats (like 4-bit integers). A 4-bit model uses less memory because each weight is stored with only 4 bits instead of 16 or 32, which drastically cuts the total storage size—since 4 bits represent a much smaller range of values, the data requires far fewer bytes to store. However, this lower precision introduces a trade-off: while the model becomes more efficient and faster to run, it may lose some accuracy because the reduced number of bits limits the granularity of the numerical representations, potentially causing slight degradation in performance compared to the original high-precision model.
Prompt: 419 tokens, 140.735 tokens-per-sec
Generation: 162 tokens, 34.874 tokens-per-sec
Peak memory: 3.685 GB

Alchemist's explanation:
Model quantization is a technique tha

## 7. Observe reasoning on an exact counting task

For this third interaction, thinking mode is enabled so you can observe how Alchemist approaches a small task that rewards careful decomposition and verification. The sentence is intentionally Star Wars-inspired and contains repeated instances of the same letter. Thinking mode uses Alchemist's recommended sampling settings so it can complete its reasoning instead of becoming trapped in repetitive greedy verification. A visible reasoning trace is useful for inspection, but it should be treated as a model-generated explanation rather than a guaranteed record of every internal computation.

In [6]:
from mlx_lm.sample_utils import make_logits_processors, make_sampler

reasoning_question = (
    "Count how many times the letter 'r' appears in the exact sentence below, "
    "ignoring uppercase and lowercase. Think briefly: count word by word, verify the "
    "arithmetic once, then give the final total.\n\n"
    "R2-D2 races through space as rebels search for Vader."
)

prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": reasoning_question}],
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True,
)

reasoning_response = generate(
    model,
    tokenizer,
    prompt=prompt,
    max_tokens=4096,
    sampler=make_sampler(temp=0.85, top_p=0.95, top_k=20),
    logits_processors=make_logits_processors(
        presence_penalty=1.1,
        presence_context_size=100000,
    ),
    verbose=True,
)

print("\nAlchemist's reasoning and final answer:")
print(reasoning_response)

Thinking Process:

1.  **Analyze the Request:**
    *   Task: Count the occurrences of the letter 'r' (case-insensitive) in the provided sentence.
    *   Sentence: "R2-D2 races through space as rebels search for Vader."
    *   Instructions: Think briefly, count word by word, verify arithmetic once, then give the final total.
    *   Role: Agent-A1 (The Alchemist).
    *   Constraints: No tools needed for this simple task (Daily Chat & Simple Questions category). Respond directly and naturally.

2.  **Analyze the Sentence:**
    *   Sentence: "R2-D2 races through space as rebels search for Vader."

3.  **Process - Word by Word:**
    *   Word 1: "R2-D2" -> Contains 'R'. Count = 1. (Note: The prompt says ignore case, so 'R' counts. '2' and '-' are not letters).
    *   Word 2: "races" -> Contains 'r', 'r'? Let's check. r-a-c-e-s. One 'r' at the start. Count = 1. Total so far = 2.
    *   Word 3: "through" -> t-h-r-o-u-g-h. One 'r'. Count = 1. Total so far = 3.
    *   Word 4: "space" -

## 8. From model output to system result

### A minimal Completion-Recovery Harness

A model can solve a task inside its reasoning trace and still fail to produce an answer that an application can deliver. This section demonstrates that distinction with an original state-tracking problem inspired by the *Tracking Shuffled Objects* family of reasoning tasks.

The baseline deliberately uses greedy decoding: the same model, prompt, runtime and device follow the same decoding path. On the reference run, Alchemist correctly tracks all five exchanges but repeatedly verifies its work until the 4,096-token budget is exhausted inside the reasoning block. The weights are not changed. Instead, a small external policy detects the incomplete termination and conditionally opens one clean answer step.

The question is written in Spanish to demonstrate that the policy is independent of the task language.

In [7]:
# Keep the exact tested prompt byte-for-byte: greedy decoding is sensitive not
# only to meaning, but also to punctuation and layout.
state_tracking_question = (
    "Cinco droides llevan un objeto diferente cada uno. "
    "Al principio: R2-D2 lleva el mapa estelar; C-3PO lleva la llave de acceso; "
    "BB-8 lleva la baliza de rastreo; K-2SO lleva el cilindro de códigos; "
    "Chopper lleva el holocrón. Después intercambian sus objetos en este orden: "
    "1. R2-D2 y BB-8. 2. C-3PO y K-2SO. 3. BB-8 y Chopper. "
    "4. K-2SO y R2-D2. 5. Chopper y C-3PO. Determina qué objeto lleva cada "
    "droide al final. Verifica cuidadosamente cada intercambio y termina con "
    "una correspondencia concisa."
)

print(state_tracking_question)

Cinco droides llevan un objeto diferente cada uno. Al principio: R2-D2 lleva el mapa estelar; C-3PO lleva la llave de acceso; BB-8 lleva la baliza de rastreo; K-2SO lleva el cilindro de códigos; Chopper lleva el holocrón. Después intercambian sus objetos en este orden: 1. R2-D2 y BB-8. 2. C-3PO y K-2SO. 3. BB-8 y Chopper. 4. K-2SO y R2-D2. 5. Chopper y C-3PO. Determina qué objeto lleva cada droide al final. Verifica cuidadosamente cada intercambio y termina con una correspondencia concisa.


![State table showing the initial item held by each of five droids, all five swaps, and the final assignment.](https://raw.githubusercontent.com/angelgalvisc/alchemist-rlm/c01c09f/docs/assets/masterclass/five-droid-swap-tracking-en.png)

*Figure 1. Deterministic state tracking for the five-droid task. Each row represents the complete state after one swap. The final assignment matches the answer delivered by the Completion-Recovery Harness.*


## 9. Formal execution contract

Let the first model step be

$$y_1 = M(x; c_r).$$

The notation has four parts:

- $M$ is the unchanged language model.
- $x$ is the original user question.
- $c_r$ is the reasoning configuration: thinking enabled, greedy decoding and a 4,096-token limit.
- $y_1$ is the complete observable output of that step, including its reasoning text and termination metadata.

The harness does **not** judge whether the droid mapping is true. It evaluates only whether the output is structurally deliverable. Define

$$D(y)=L(y)\land A(y)\land N(y),$$

where:

- $L(y)$ is true when the reasoning block is complete. For this model adapter, that means `</think>` is present.
- $A(y)$ is true when non-empty answer content exists after the reasoning block.
- $N(y)$ is true when generation stopped normally rather than because the token limit was reached.

Thus $D(y)=1$ means *deliverable*, not necessarily *correct*. Correctness and deliverability remain separate measurements.

The bounded recovery output is

$$
y_2=M(R(x,y_1);c_f).
$$

The delivery harness policy is

$$
H_D(M,x)=
\begin{cases}
y_1, & D(y_1)=1,\\[4pt]
y_2, & D(y_1)=0 \land D(y_2)=1,\\[4pt]
\bot, & \text{otherwise}.
\end{cases}
$$

Here $H_D$ is the delivery harness, $R(x,y_1)$ constructs a recovery request from the original question and the untouched draft, and $c_f$ is a short-answer configuration with thinking disabled. The policy allows at most one recovery step and returns explicit failure if that step also violates the delivery contract.

In plain language: **Detect → Steer → Recover**.

## 10. Implement the harness in plain Python

The implementation below deliberately avoids a framework. `run_step` performs one model request, `inspect_completion` applies the structural contract, and `completion_recovery` controls the conditional second step. The model-specific detail—how a closed reasoning block is recognized—is isolated in the inspector rather than spread through the policy.

In [8]:
from mlx_lm import stream_generate


def run_step(user_content, *, thinking, max_tokens):
    """Run one greedy model step and retain its termination metadata."""
    formatted_prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": user_content}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=thinking,
    )

    fragments = []
    last_event = None
    for event in stream_generate(
        model, tokenizer, prompt=formatted_prompt, max_tokens=max_tokens
    ):
        fragments.append(event.text)
        last_event = event

    if last_event is None:
        raise RuntimeError("The model produced no generation events.")

    return {
        "text": "".join(fragments),
        "thinking": thinking,
        "tokens": last_event.generation_tokens,
        "tokens_per_second": last_event.generation_tps,
        "finish_reason": last_event.finish_reason,
    }


def inspect_completion(step):
    """Measure deliverability, not semantic correctness."""
    text = step["text"]

    if step["thinking"]:
        reasoning_closed = "</think>" in text
        final_content = (
            text.rsplit("</think>", 1)[1].strip() if reasoning_closed else ""
        )
    else:
        reasoning_closed = True
        final_content = text.strip()

    stopped_normally = step["finish_reason"] == "stop"
    deliverable = reasoning_closed and bool(final_content) and stopped_normally

    return {
        "reasoning_closed": reasoning_closed,
        "has_final_content": bool(final_content),
        "stopped_normally": stopped_normally,
        "deliverable": deliverable,
        "final_content": final_content,
    }


def completion_recovery(question, *, reasoning_budget=4096, answer_budget=512):
    """Return a deliverable answer after at most one recovery step."""
    first = run_step(question, thinking=True, max_tokens=reasoning_budget)
    first_check = inspect_completion(first)

    if first_check["deliverable"]:
        return {
            "answer": first_check["final_content"],
            "recovery_used": False,
            "first_step": first,
            "first_check": first_check,
            "recovery_step": None,
            "recovery_check": None,
        }

    recovery_request = f"""{question}

A continuación aparece el borrador bruto de un intento anterior. Puede estar truncado, ser repetitivo o no haber cerrado su bloque de razonamiento. No asumas que su formato es correcto.

--- BORRADOR ---
{first['text']}
--- FIN DEL BORRADOR ---

Responde la pregunta original usando la información válida del borrador. Entrega solamente la correspondencia final, una línea por droide, sin razonamiento, introducción ni conclusión."""

    recovery = run_step(
        recovery_request, thinking=False, max_tokens=answer_budget
    )
    recovery_check = inspect_completion(recovery)

    if not recovery_check["deliverable"]:
        raise RuntimeError(
            "The single permitted recovery step was not deliverable."
        )

    return {
        "answer": recovery_check["final_content"],
        "recovery_used": True,
        "first_step": first,
        "first_check": first_check,
        "recovery_step": recovery,
        "recovery_check": recovery_check,
    }

## 11. Run the controlled experiment

This call executes the complete policy. The first step is the model-only baseline. If it is not deliverable, the harness activates exactly one clean step. Expect the baseline to take longer than the recovery because it is allowed to consume up to 4,096 tokens.

In [9]:
harness_result = completion_recovery(state_tracking_question)

first = harness_result["first_step"]
check = harness_result["first_check"]

print("MODEL-ONLY STEP")
print("-----------------")
print(f"Generated tokens:     {first['tokens']}")
print(f"Tokens per second:    {first['tokens_per_second']:.2f}")
print(f"Finish reason:        {first['finish_reason']}")
print(f"Reasoning closed:     {check['reasoning_closed']}")
print(f"Final content found:  {check['has_final_content']}")
print(f"Deliverable:          {check['deliverable']}")
print(f"Recovery activated:   {harness_result['recovery_used']}")

# Show enough of the raw draft to make repeated verification visible without
# flooding the notebook. The complete 4,096-token draft remains auditable as
# harness_result["first_step"]["text"].
preview_chars = 1800
print("\nTAIL OF THE RAW REASONING DRAFT")
print("--------------------------------")
print(first["text"][-preview_chars:])

if harness_result["recovery_used"]:
    recovery = harness_result["recovery_step"]
    recovery_check = harness_result["recovery_check"]
    print("\nRECOVERY STEP")
    print("-------------")
    print(f"Generated tokens:     {recovery['tokens']}")
    print(f"Tokens per second:    {recovery['tokens_per_second']:.2f}")
    print(f"Finish reason:        {recovery['finish_reason']}")
    print(f"Deliverable:          {recovery_check['deliverable']}")

print("\nFINAL DELIVERABLE ANSWER")
print("------------------------")
print(harness_result["answer"])

MODEL-ONLY STEP
-----------------
Generated tokens:     4096
Tokens per second:    31.53
Finish reason:        length
Reasoning closed:     False
Final content found:  False
Deliverable:          False
Recovery activated:   True

TAIL OF THE RAW REASONING DRAFT
--------------------------------
ay, final output generation.

    Wait, I should check the date. Current date: 2026-08-19. This is irrelevant for the puzzle but good to keep in mind if needed. Not needed here.

    Okay, generating response.

    Wait, I should check if there is any ambiguity about "intercambian sus objetos en este orden". Does it mean they swap simultaneously or sequentially? "En este orden" usually implies sequential steps 1, 2, 3... So my sequential assumption is correct.

    Okay, ready.

    Final check on Spanish grammar.
    "Llave de acceso" - correct.
    "Mapa estelar" - correct.
    "Baliza de rastreo" - correct.
    "Cilindro de códigos" - correct.
    "Holocrón" - correct.

    Okay.

    Let's pr

## 12. What the harness changed—and what it did not

The first step and the recovery step use the same model weights. The harness does not contain the correct droid mapping, does not edit the reasoning draft and does not perform semantic grading. Its contribution is execution policy:

1. preserve the model's complete observable draft;
2. inspect whether the output satisfies a delivery contract;
3. oppose termination when the contract is not satisfied;
4. open one bounded answer step with thinking disabled; and
5. fail explicitly rather than recurse forever if recovery is unsuccessful.

This distinction is the central lesson: model capability and system reliability are related but not identical. The model may contain enough information to solve a task while the surrounding loop determines whether that information becomes a usable result.

The experiment demonstrates **completion recovery**, not proof of semantic correctness and not recursive language modeling. A production harness can add independent semantic validators, tool use, persistence or recursion as separate policies.

## 13. Quantization and behavioral stability

### Comparing Alchemist with its unquantized parent

[Agents-A1-4B](https://huggingface.co/InternScience/Agents-A1-4B) is the official 16-bit parent checkpoint from which [Alchemist 4-bit](https://huggingface.co/angelgalvisc/agent-a1-alchemist-4bit) was quantized. The tested lineage is **Agents-A1-4B (BF16) → 4-bit quantization → Alchemist**.

The published weight files occupy approximately **9.08 GB** for Agents and **3.06 GB** for Alchemist: a **66.3% reduction**, or about **2.97× smaller**. On the Kaggle T4, Agents is executed in FP16 because T4 does not provide native BF16 tensor-core execution. This BF16-to-FP16 execution conversion keeps the parent at 16-bit precision; it is not 4-bit quantization.

The following cell is intentionally self-contained, so this comparison can be run without rerunning the earlier Alchemist demonstrations. It uses the exact same Spanish state-tracking prompt, chat-template thinking mode, greedy decoding and 4,096-token ceiling used for Alchemist. The full generated trace is retained and printed, together with termination metadata and peak GPU memory.

In [10]:
# Self-contained Agents-A1-4B comparison: only this cell needs to be run.
%pip install -q --upgrade "mlx[cuda13]==0.32.1" "mlx-lm==0.31.3"

import gc
import mlx.core as mx
from mlx_lm import load, stream_generate
from mlx_lm.sample_utils import make_sampler

for name in ("model", "tokenizer"):
    if name in globals():
        del globals()[name]
gc.collect()
mx.clear_cache()
mx.reset_peak_memory()

agents_model, agents_tokenizer = load("InternScience/Agents-A1-4B")
# T4-compatible 16-bit execution; this does not quantize the parent checkpoint.
agents_model.set_dtype(mx.float16)
mx.eval(agents_model.parameters())
mx.reset_peak_memory()

state_tracking_question = (
    "Cinco droides llevan un objeto diferente cada uno. "
    "Al principio: R2-D2 lleva el mapa estelar; C-3PO lleva la llave de acceso; "
    "BB-8 lleva la baliza de rastreo; K-2SO lleva el cilindro de códigos; "
    "Chopper lleva el holocrón. Después intercambian sus objetos en este orden: "
    "1. R2-D2 y BB-8. 2. C-3PO y K-2SO. 3. BB-8 y Chopper. "
    "4. K-2SO y R2-D2. 5. Chopper y C-3PO. Determina qué objeto lleva cada "
    "droide al final. Verifica cuidadosamente cada intercambio y termina con "
    "una correspondencia concisa."
)

agents_prompt = agents_tokenizer.apply_chat_template(
    [{"role": "user", "content": state_tracking_question}],
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True,
)

parts = []
last_event = None
for event in stream_generate(
    agents_model,
    agents_tokenizer,
    prompt=agents_prompt,
    max_tokens=4096,
    sampler=make_sampler(temp=0.0),
):
    parts.append(event.text)
    last_event = event

if last_event is None:
    raise RuntimeError("Agents-A1-4B produced no generation events.")

agents_text = "".join(parts)
reasoning_closed = "</think>" in agents_text
final_text = agents_text.split("</think>", 1)[1].strip() if reasoning_closed else ""

print("AGENTS-A1-4B — ORIGINAL 16-BIT PARENT")
print("--------------------------------------")
print(f"Generated tokens:     {last_event.generation_tokens}")
print(f"Tokens per second:    {last_event.generation_tps:.2f}")
print(f"Peak GPU memory:      {last_event.peak_memory:.3f} GB")
print(f"Finish reason:        {last_event.finish_reason}")
print(f"Reasoning closed:     {reasoning_closed}")
print(f"Final content found:  {bool(final_text)}")

print("\nFULL GENERATED TRACE")
print("--------------------")
print(agents_text)

print("\nCONTROLLED COMPARISON")
print("---------------------")
print("Alchemist 4-bit: 3.06 GB weights | 4096 tokens | length | no deliverable")
print(
    f"Agents 16-bit:   9.08 GB weights | {last_event.peak_memory:.3f} GB peak | "
    f"{last_event.generation_tokens} tokens | {last_event.finish_reason} | "
    f"{'deliverable' if final_text else 'no deliverable'}"
)

Note: you may need to restart the kernel to use updated packages.


You are using a model of type qwen3_5 to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


AGENTS-A1-4B — ORIGINAL 16-BIT PARENT
--------------------------------------
Generated tokens:     2402
Tokens per second:    26.25
Peak GPU memory:      9.678 GB
Finish reason:        stop
Reasoning closed:     True
Final content found:  True

FULL GENERATED TRACE
--------------------
Thinking Process:

1.  **Analyze the Request:**
    *   **Task:** Determine which object each droid carries at the end after a series of exchanges.
    *   **Initial State:**
        *   R2-D2: Star Map (Mapa estelar)
        *   C-3PO: Access Key (Llave de acceso)
        *   BB-8: Tracking Beacon (Baliza de rastreo)
        *   K-2SO: Code Cylinder (Cilindro de códigos)
        *   Chopper: Holocron (Holocrón)
    *   **Exchanges (in order):**
        1.  R2-D2 and BB-8
        2.  C-3PO and K-2SO
        3.  BB-8 and Chopper
        4.  K-2SO and R2-D2
        5.  Chopper and C-3PO
    *   **Output Requirement:** Carefully verify each exchange and end with a concise correspondence.
    *   **Language:

### Interpretation

Under the matched decoding contract, the 16-bit parent completed the task in **2,402 tokens at 26.25 tokens/s**, stopped naturally, closed its reasoning block and returned a final answer. Its measured peak GPU memory was **9.678 GB**. Alchemist's first unassisted attempt reached the 4,096-token ceiling without closing the reasoning block or exposing a final answer; the recovery step then produced a deliverable in 47 tokens.

This single controlled task does **not** prove that quantization alone caused the loop: checkpoint packaging, numerical execution and task sensitivity may also contribute. It does show a behaviorally meaningful difference consistent with reduced-precision inference affecting termination stability. The Completion-Recovery Harness is therefore useful at the system boundary: it converts an otherwise non-deliverable run into a bounded final response. The harness does not reconstruct the 16-bit weights or claim to restore every behavior of the parent model.


## Installation and harness demonstration complete

Alchemist is now running with MLX on a Kaggle T4 GPU. You have tested direct generation, inspected an optional reasoning trace, and executed a minimal Completion-Recovery Harness. The same initialized model and tokenizer can now be connected to the full Alchemist-RLM Harness for long-context and agentic workflows.

## 14. Applied example: from an everyday question to a verifiable result

A person does not formulate filters, operations or structured queries. They simply ask:

> **Oye, ¿cuánto me gasté en servicios públicos el mes pasado?**

The question remains verbatim in both execution paths. The model weights and transaction set are also unchanged; what differs is the surrounding execution policy.

| ID | Date | Transaction | Amount | Status |
|---|---|---|---:|---|
| `M001` | 02 Jul | SUPERMERCADO ÉXITO | $84,650 | Approved |
| `M002` | 03 Jul | ENEL COLOMBIA | $167,420 | Approved |
| `M003` | 03 Jul | PSE ENEL COLOMBIA | $167,420 | Rejected |
| `M004` | 05 Jul | UBER | $18,900 | Approved |
| `M005` | 07 Jul | VANTI GAS NATURAL | $61,380 | Approved |
| `M006` | 08 Jul | PAN PA' YA | $16,700 | Approved |
| `M007` | 10 Jul | CLARO HOGAR | $112,900 | Approved |
| `M008` | 12 Jul | ACUEDUCTO DE BOGOTÁ | $96,750 | Approved |
| `M009` | 12 Jul | PSE ACUEDUCTO BOGOTÁ | $96,750 | Rejected |
| `M010` | 14 Jul | RECARGA NEQUI | $30,000 | Approved |
| `M011` | 17 Jul | RESTAURANTE CREPES | $57,800 | Approved |
| `M012` | 19 Jul | VANTI GAS NATURAL | $61,380 | Rejected |
| `M013` | 21 Jul | NETFLIX | $26,900 | Approved |
| `M014` | 23 Jul | ENVÍO A CAMILA | $75,000 | Approved |
| `M015` | 25 Jul | TRANSMILENIO | $20,000 | Approved |
| `M016` | 28 Jul | FARMATODO | $43,250 | Approved |
| `M017` | 30 Jul | TIENDAS D1 | $71,320 | Approved |

### Execution contract

Let:

- $q$ be the user's question;
- $T$ be the transaction set;
- $M$ be the local model;
- $\Sigma$ be the allowed operations, categories and statuses;
- $p$ be a plan proposed by the model;
- $V(p)$ validate that plan; and
- $E(T,p)$ execute it deterministically.

A direct response asks the model to interpret, filter and calculate in one generation:

$$
y_{\mathrm{direct}} = M(q,T).
$$

The harness first asks for a plan:

$$
p_1=M(q,\Sigma).
$$

Its bounded policy is

$$
H_V(M,q,T)=
\begin{cases}
E(T,p_1), & V(p_1)=1,\\[4pt]
E(T,p_2), & V(p_1)=0 \land V(p_2)=1,\\[4pt]
\bot, & \text{otherwise},
\end{cases}
$$

where

$$
p_2=M\bigl(R(q,p_1),\Sigma\bigr).
$$

$H_V$ is the validated-execution harness. $R$ is one bounded repair request. $\bot$ means explicit failure instead of an unverifiable answer. The recorded output below reports whether the bounded repair path was activated.

The next cell is self-contained. It loads Alchemist once, sends the exact same natural-language question through both paths, preserves both observable outputs and executes only validated arithmetic in Python.


In [3]:
# Self-contained applied example: run only this cell.
%pip install -q --upgrade "mlx[cuda13]==0.32.1" "mlx-lm==0.31.3"

import gc
import json
import re
import mlx.core as mx
from mlx_lm import load, generate

# Release another model if this cell is run in an existing session.
for name in ("model", "tokenizer", "agents_model", "agents_tokenizer"):
    if name in globals():
        del globals()[name]
gc.collect()
mx.clear_cache()

model, tokenizer = load("angelgalvisc/agent-a1-alchemist-4bit")
model.set_dtype(mx.float16)
mx.eval(model.parameters())

question = "Oye, ¿cuánto me gasté en servicios públicos el mes pasado?"
transactions = [
    {"id":"M001","date":"2026-07-02","merchant":"SUPERMERCADO ÉXITO","amount":84650,"status":"approved","category":"groceries"},
    {"id":"M002","date":"2026-07-03","merchant":"ENEL COLOMBIA","amount":167420,"status":"approved","category":"electricity"},
    {"id":"M003","date":"2026-07-03","merchant":"PSE ENEL COLOMBIA","amount":167420,"status":"rejected","category":"electricity"},
    {"id":"M004","date":"2026-07-05","merchant":"UBER","amount":18900,"status":"approved","category":"transport"},
    {"id":"M005","date":"2026-07-07","merchant":"VANTI GAS NATURAL","amount":61380,"status":"approved","category":"gas"},
    {"id":"M006","date":"2026-07-08","merchant":"PAN PA' YA","amount":16700,"status":"approved","category":"restaurants"},
    {"id":"M007","date":"2026-07-10","merchant":"CLARO HOGAR","amount":112900,"status":"approved","category":"internet"},
    {"id":"M008","date":"2026-07-12","merchant":"ACUEDUCTO DE BOGOTÁ","amount":96750,"status":"approved","category":"water"},
    {"id":"M009","date":"2026-07-12","merchant":"PSE ACUEDUCTO BOGOTÁ","amount":96750,"status":"rejected","category":"water"},
    {"id":"M010","date":"2026-07-14","merchant":"RECARGA NEQUI","amount":30000,"status":"approved","category":"top_up"},
    {"id":"M011","date":"2026-07-17","merchant":"RESTAURANTE CREPES","amount":57800,"status":"approved","category":"restaurants"},
    {"id":"M012","date":"2026-07-19","merchant":"VANTI GAS NATURAL","amount":61380,"status":"rejected","category":"gas"},
    {"id":"M013","date":"2026-07-21","merchant":"NETFLIX","amount":26900,"status":"approved","category":"subscriptions"},
    {"id":"M014","date":"2026-07-23","merchant":"ENVÍO A CAMILA","amount":75000,"status":"approved","category":"transfers"},
    {"id":"M015","date":"2026-07-25","merchant":"TRANSMILENIO","amount":20000,"status":"approved","category":"transport"},
    {"id":"M016","date":"2026-07-28","merchant":"FARMATODO","amount":43250,"status":"approved","category":"health"},
    {"id":"M017","date":"2026-07-30","merchant":"TIENDAS D1","amount":71320,"status":"approved","category":"groceries"},
]

allowed_operations = {"sum", "count", "list"}
allowed_statuses = {"approved", "rejected", "reversed"}
allowed_categories = {tx["category"] for tx in transactions}

def prompt_for(content):
    return tokenizer.apply_chat_template(
        [{"role":"user", "content":content}], tokenize=False,
        add_generation_prompt=True, enable_thinking=False,
    )

def ask(content, max_tokens):
    return generate(model, tokenizer, prompt=prompt_for(content),
                    max_tokens=max_tokens, verbose=False).strip()

def ledger_text():
    lines = ["Estas son mis transacciones de julio de 2026:"]
    status_es = {"approved":"Aprobada", "rejected":"Rechazada"}
    for tx in transactions:
        amount = "$" + f'{tx["amount"]:,}'.replace(",", ".")
        lines.append(
            f'{tx["id"]} | {tx["date"]} | {tx["merchant"]} | '
            f'{amount} | {status_es[tx["status"]]}'
        )
    return "\n".join(lines)

def extract_json(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        raise ValueError("The model did not return a JSON plan.")
    return json.loads(match.group(0))

def valid(plan):
    filters = plan.get("filters", {})
    return (
        plan.get("operation") in allowed_operations
        and set(filters.get("categories", [])) <= allowed_categories
        and set(filters.get("status", [])) <= allowed_statuses
        and isinstance(filters.get("date_from"), str)
        and isinstance(filters.get("date_to"), str)
    )

def execute(plan):
    filters = plan["filters"]
    selected = [
        tx for tx in transactions
        if filters["date_from"] <= tx["date"] <= filters["date_to"]
        and tx["category"] in filters["categories"]
        and tx["status"] in filters["status"]
    ]
    operation = plan["operation"]
    value = (sum(tx["amount"] for tx in selected) if operation == "sum"
             else len(selected) if operation == "count" else selected)
    return {"value": value, "transactions": selected}

def pesos(value):
    return "$" + f"{value:,}".replace(",", ".")

# Path A: the model interprets, filters and calculates in one generation.
direct_input = ledger_text() + "\n\n" + question
if "direct_answer" not in globals():
    direct_answer = ask(direct_input, 1024)

# Path B: the same question becomes a validated executable plan.
contract = f"""
Convierte una pregunta sobre movimientos financieros en un plan JSON ejecutable.
Hoy es 2026-08-19.
Contrato: {{"operation":"<one operation>","filters":{{"date_from":"YYYY-MM-DD","date_to":"YYYY-MM-DD","categories":["..."],"status":["..."]}}}}
Elige operation como exactamente un valor de ["sum", "count", "list"].
Elige cada status únicamente de ["approved", "rejected", "reversed"].
Categorías disponibles: {sorted(allowed_categories)}.
Política del producto: "servicios públicos" significa electricity, gas y water;
internet y top_up son categorías distintas. Un gasto efectivo debe estar approved.
Devuelve solo el JSON, sin calcular valores ni seleccionar IDs manualmente.
Pregunta: {question}
""".strip()
raw_plan = ask(contract, 512)
plan = extract_json(raw_plan)
repair_used = False

if not valid(plan):
    repair_used = True
    repair = contract + "\nEl plan anterior no cumplió el contrato. Elige un solo valor permitido para operation y devuelve únicamente un plan JSON válido."
    raw_plan = ask(repair, 512)
    plan = extract_json(raw_plan)
if not valid(plan):
    raise RuntimeError("The bounded repair did not produce a valid plan.")

result = execute(plan)
assert question in direct_input and question in contract

print("SAME USER QUESTION IN BOTH PATHS")
print("--------------------------------")
print(question)
print("\nMODEL-ONLY ANSWER")
print("-----------------")
print(direct_answer)
print("\nMODEL PLAN")
print("----------")
print(json.dumps(plan, ensure_ascii=False, indent=2))
print(f"\nContract valid: {valid(plan)}")
print(f"Repair used:   {repair_used}")
print("\nDETERMINISTIC EXECUTION")
print("-----------------------")
for tx in result["transactions"]:
    print(f'✓ {tx["id"]}  {tx["merchant"]:<24} {pesos(tx["amount"]):>10}')
print(f'\nTOTAL: {pesos(result["value"])}')


SAME USER QUESTION IN BOTH PATHS
--------------------------------
Oye, ¿cuánto me gasté en servicios públicos el mes pasado?

MODEL-ONLY ANSWER
-----------------
Para determinar cuánto gastaste en servicios públicos en julio de 2026, necesitamos identificar las transacciones relacionadas con servicios públicos (como luz, agua, gas, teléfono, etc.) y sumar solo las que fueron **aprobadas**.

Aquí están las transacciones que corresponden a servicios públicos:

1. **M002 | 2026-07-03 | ENEL COLOMBIA | $167.420 | Aprobada** → Luz (aprobada)
2. **M003 | 2026-07-03 | PSE ENEL COLOMBIA | $167.420 | Rechazada** → Luz (rechazada, no cuenta)
3. **M005 | 2026-07-07 | VANTI GAS NATURAL | $61.380 | Aprobada** → Gas (aprobada)
4. **M006 | 2026-07-08 | PAN PA' YA | $16.700 | Aprobada** → No es un servicio público (es comida)
5. **M007 | 2026-07-10 | CLARO HOGAR | $112.900 | Aprobada** → Teléfono/Internet (aprobada)
6. **M008 | 2026-07-12 | ACUEDUCTO DE BOGOTÁ | $96.750 | Aprobada** → Agua (aprobada)


### What happened in this Kaggle run

The preceding cell preserves the complete observable output of both paths. They used the same Alchemist checkpoint, the same natural-language question and the same transaction set.

| Stage | Without harness | With harness |
|---|---|---|
| Natural-language input | Same question | Same question |
| Model responsibility | Interpret, select and calculate | Produce an executable plan |
| Observable intermediate result | Free-form answer | Validatable JSON plan |
| Selection | Included CLARO HOGAR and TRANSMILENIO | Approved electricity, gas and water payments |
| Arithmetic | Performed by the model | Performed by Python |
| Result | **$458,450** | **$325,550** |
| Contract validation | Not available | `True` |
| Repair | Not applicable | Not used |

#### Direct path

The model attempted to complete the entire task in one generation. It correctly excluded the rejected transactions, but classified `CLARO HOGAR` and `TRANSMILENIO` as public utilities and reported **$458,450**.

#### Harness path

The model did not calculate the result. It produced a plan specifying the date range, the categories `electricity`, `gas` and `water`, and the status `approved`. The plan passed validation on its first attempt. Python then selected `M002`, `M005` and `M008` and calculated **$325,550**.

```text
SAME QUESTION
      │
      ├── Direct model ──→ free selection ──→ $458,450
      │
      └── Harness ──→ JSON plan ──→ validation ──→ Python ──→ $325,550
```

The JSON plan is an observable operational trace: it records what the system chose to execute without requiring access to private chain-of-thought.

> **The direct model produced an answer. The harness produced a verifiable result.**


## 15. Managing Reasoning Loops: Observability and Recovery with a Harness

A harness can preserve and use the valuable work contained in a reasoning trajectory even when the generation does not complete. This section shows how to observe that state, identify verifiable intermediate results and convert them into a controlled system decision.

We repeat the same question, transaction set and Alchemist checkpoint under a single reasoning configuration $c_r$: thinking enabled, greedy decoding and a maximum of 4,096 tokens per path.

> **Oye, ¿cuánto me gasté en servicios públicos el mes pasado?**

The direct path asks the model to interpret, select, calculate and answer. The harness path asks it to produce an executable plan under an explicit contract. Both observable traces are retained. The system does not grade individual thoughts; it checks whether a trace contains an independently verifiable intermediate state.

Let $\tau$ be an observable trace, $\Sigma$ the plan contract, $V(p)$ its validator and $E(T,p)$ the deterministic executor. The two matched-configuration traces are

$$
\tau_{\mathrm{direct}}=M(q,T;c_r),
\qquad
\tau_{\mathrm{plan}}=M(q,\Sigma;c_r).
$$

The set of schema-valid JSON candidates observable in a trace is

$$
C(\tau)=\left\{p\in\operatorname{JSON}(\tau):V(p)=1\right\}.
$$

Structural recovery is defined as

$$
S(\tau,T)=
\begin{cases}
E(T,p), & C(\tau)=\{p\},\\[4pt]
\bot, & |C(\tau)|\neq 1.
\end{cases}
$$

The harness proceeds only when exactly one distinct observable candidate satisfies the independent contract; otherwise it fails explicitly.

The next cell is self-contained for a fresh run. In this recorded update, it reuses the existing 4,096-token direct trace from the active Kaggle session and executes only the harness path at the same 4,096-token budget. No earlier notebook cell is rerun.


In [1]:
# Self-contained thinking comparison: run only this cell.
%pip install -q --upgrade "mlx[cuda13]==0.32.1" "mlx-lm==0.31.3"

import gc
import json
from pathlib import Path
import mlx.core as mx
from mlx_lm import load, stream_generate
from mlx_lm.sample_utils import make_sampler

for name in ("model", "tokenizer", "agents_model", "agents_tokenizer"):
    if name in globals():
        del globals()[name]
gc.collect()
mx.clear_cache()

model, tokenizer = load("angelgalvisc/agent-a1-alchemist-4bit")
model.set_dtype(mx.float16)
mx.eval(model.parameters())

question = "Oye, ¿cuánto me gasté en servicios públicos el mes pasado?"
transactions = [
    {"id":"M001","date":"2026-07-02","merchant":"SUPERMERCADO ÉXITO","amount":84650,"status":"approved","category":"groceries"},
    {"id":"M002","date":"2026-07-03","merchant":"ENEL COLOMBIA","amount":167420,"status":"approved","category":"electricity"},
    {"id":"M003","date":"2026-07-03","merchant":"PSE ENEL COLOMBIA","amount":167420,"status":"rejected","category":"electricity"},
    {"id":"M004","date":"2026-07-05","merchant":"UBER","amount":18900,"status":"approved","category":"transport"},
    {"id":"M005","date":"2026-07-07","merchant":"VANTI GAS NATURAL","amount":61380,"status":"approved","category":"gas"},
    {"id":"M006","date":"2026-07-08","merchant":"PAN PA' YA","amount":16700,"status":"approved","category":"restaurants"},
    {"id":"M007","date":"2026-07-10","merchant":"CLARO HOGAR","amount":112900,"status":"approved","category":"internet"},
    {"id":"M008","date":"2026-07-12","merchant":"ACUEDUCTO DE BOGOTÁ","amount":96750,"status":"approved","category":"water"},
    {"id":"M009","date":"2026-07-12","merchant":"PSE ACUEDUCTO BOGOTÁ","amount":96750,"status":"rejected","category":"water"},
    {"id":"M010","date":"2026-07-14","merchant":"RECARGA NEQUI","amount":30000,"status":"approved","category":"top_up"},
    {"id":"M011","date":"2026-07-17","merchant":"RESTAURANTE CREPES","amount":57800,"status":"approved","category":"restaurants"},
    {"id":"M012","date":"2026-07-19","merchant":"VANTI GAS NATURAL","amount":61380,"status":"rejected","category":"gas"},
    {"id":"M013","date":"2026-07-21","merchant":"NETFLIX","amount":26900,"status":"approved","category":"subscriptions"},
    {"id":"M014","date":"2026-07-23","merchant":"ENVÍO A CAMILA","amount":75000,"status":"approved","category":"transfers"},
    {"id":"M015","date":"2026-07-25","merchant":"TRANSMILENIO","amount":20000,"status":"approved","category":"transport"},
    {"id":"M016","date":"2026-07-28","merchant":"FARMATODO","amount":43250,"status":"approved","category":"health"},
    {"id":"M017","date":"2026-07-30","merchant":"TIENDAS D1","amount":71320,"status":"approved","category":"groceries"},
]
allowed_operations = {"sum", "count", "list"}
allowed_statuses = {"approved", "rejected", "reversed"}
allowed_categories = {tx["category"] for tx in transactions}

def pesos(value):
    return "$" + f"{value:,}".replace(",", ".")

def ledger_text():
    status_es = {"approved":"Aprobada", "rejected":"Rechazada"}
    lines = ["Estas son mis transacciones de julio de 2026:"]
    for tx in transactions:
        lines.append(
            f'{tx["id"]} | {tx["date"]} | {tx["merchant"]} | '
            f'{pesos(tx["amount"])} | {status_es[tx["status"]]}'
        )
    return "\n".join(lines)

def prompt_for(content):
    return tokenizer.apply_chat_template(
        [{"role":"user", "content":content}], tokenize=False,
        add_generation_prompt=True, enable_thinking=True,
    )

def run_trace(content, max_tokens):
    parts, last = [], None
    for event in stream_generate(
        model, tokenizer, prompt=prompt_for(content),
        max_tokens=max_tokens, sampler=make_sampler(temp=0.0),
    ):
        parts.append(event.text)
        last = event
    if last is None:
        raise RuntimeError("The model produced no generation events.")
    return "".join(parts), last

def valid(plan):
    filters = plan.get("filters", {})
    return (
        plan.get("operation") in allowed_operations
        and set(filters.get("categories", [])) <= allowed_categories
        and set(filters.get("status", [])) <= allowed_statuses
        and isinstance(filters.get("date_from"), str)
        and isinstance(filters.get("date_to"), str)
    )

def valid_plans(trace):
    decoder, candidates = json.JSONDecoder(), {}
    for start, char in enumerate(trace):
        if char != "{":
            continue
        try:
            candidate, _ = decoder.raw_decode(trace[start:])
        except json.JSONDecodeError:
            continue
        if isinstance(candidate, dict) and valid(candidate):
            candidates[json.dumps(candidate, sort_keys=True)] = candidate
    return list(candidates.values())

def execute(plan):
    f = plan["filters"]
    selected = [
        tx for tx in transactions
        if f["date_from"] <= tx["date"] <= f["date_to"]
        and tx["category"] in f["categories"]
        and tx["status"] in f["status"]
    ]
    operation = plan["operation"]
    value = (sum(tx["amount"] for tx in selected) if operation == "sum"
             else len(selected) if operation == "count" else selected)
    return {"value":value, "transactions":selected}

def excerpt(trace, head, tail):
    if len(trace) <= head + tail:
        return trace
    omitted = len(trace) - head - tail
    return trace[:head] + f"\n\n... [{omitted} characters omitted] ...\n\n" + trace[-tail:]

# Same natural-language question, path A: unconstrained direct reasoning.
direct_trace_reused = (
    "direct_trace" in globals() and "direct_event" in globals()
    and direct_event.generation_tokens == 4096
    and "TRANSMILENIO" in direct_trace
)
if not direct_trace_reused:
    direct_trace, direct_event = run_trace(ledger_text() + "\n\n" + question, 4096)

# Same natural-language question, path B: reasoning under a plan contract.
contract = f"""
Convierte una pregunta sobre movimientos financieros en un plan JSON ejecutable.
Hoy es 2026-08-19.
Contrato: {{"operation":"<one operation>","filters":{{"date_from":"YYYY-MM-DD","date_to":"YYYY-MM-DD","categories":["..."],"status":["..."]}}}}
Elige operation como exactamente un valor de ["sum", "count", "list"].
Elige cada status únicamente de ["approved", "rejected", "reversed"].
Categorías disponibles: {sorted(allowed_categories)}.
Política del producto: "servicios públicos" significa electricity, gas y water;
internet y top_up son categorías distintas. Un gasto efectivo debe estar approved.
Devuelve el plan JSON después de terminar tu razonamiento. No calcules valores ni selecciones IDs manualmente.
Pregunta: {question}
""".strip()
harness_trace, harness_event = run_trace(contract, 4096)

Path("/kaggle/working/direct_thinking_trace.txt").write_text(direct_trace)
Path("/kaggle/working/harness_thinking_trace.txt").write_text(harness_trace)

candidates = valid_plans(harness_trace)
if len(candidates) != 1:
    raise RuntimeError(f"Expected exactly one valid plan; found {len(candidates)}.")
plan = candidates[0]
result = execute(plan)

print("EXPERIMENT CONTRACT")
print("-------------------")
print("Question reused verbatim: True")
print("Thinking enabled:         True")
print("Decoding:                 greedy / temperature 0")
print("Budget per path:          4096 tokens")
print(f"Direct trace reused:      {direct_trace_reused}")

print("\nDIRECT THINKING METADATA")
print("------------------------")
print(f"Generated tokens:     {direct_event.generation_tokens}")
print(f"Finish reason:        {direct_event.finish_reason}")
print(f"Reasoning closed:     {'</think>' in direct_trace}")
print(f"Final answer found:   {bool(direct_trace.split('</think>',1)[1].strip()) if '</think>' in direct_trace else False}")
print("\nDIRECT TRACE — REPRESENTATIVE EXCERPT")
print("-------------------------------------")
print(excerpt(direct_trace, 3600, 1700))

print("\nHARNESS THINKING METADATA")
print("-------------------------")
print(f"Generated tokens:     {harness_event.generation_tokens}")
print(f"Finish reason:        {harness_event.finish_reason}")
print(f"Reasoning closed:     {'</think>' in harness_trace}")
print(f"Final answer found:   {bool(harness_trace.split('</think>',1)[1].strip()) if '</think>' in harness_trace else False}")
print("\nHARNESS TRACE — REPRESENTATIVE EXCERPT")
print("--------------------------------------")
print(excerpt(harness_trace, 5200, 1400))

print("\nSTRUCTURAL RECOVERY")
print("-------------------")
print(f"Distinct schema-valid plans: {len(candidates)}")
print(json.dumps(plan, ensure_ascii=False, indent=2))
print("\nDETERMINISTIC EXECUTION")
for tx in result["transactions"]:
    print(f'✓ {tx["id"]}  {tx["merchant"]:<24} {pesos(tx["amount"]):>10}')
print(f'\nRECOVERED TOTAL: {pesos(result["value"])}')
print("\nFull traces saved to /kaggle/working/direct_thinking_trace.txt and harness_thinking_trace.txt")


EXPERIMENT CONTRACT
-------------------
Question reused verbatim: True
Thinking enabled:         True
Decoding:                 greedy / temperature 0
Budget per path:          4096 tokens
Direct trace reused:      True

DIRECT THINKING METADATA
------------------------
Generated tokens:     4096
Finish reason:        length
Reasoning closed:     False
Final answer found:   False

DIRECT TRACE — REPRESENTATIVE EXCERPT
-------------------------------------
Thinking Process:

1.  **Analyze the Request:**
    *   Input: A list of transactions for July 2026 (M001 to M017).
    *   Question: "Oye, ¿cuánto me gasté en servicios públicos el mes pasado?" (Hey, how much did I spend on public services last month?)
    *   Context: The user provided a list of transactions with dates, merchant names, amounts, and status (Approved/Rejected).
    *   Task: Identify which transactions correspond to "servicios públicos" (public services) and sum their amounts.

2.  **Analyze the Transaction Data:**
  

### What the matched-budget comparison demonstrates

Fixing the generation budget at 4,096 tokens for both paths removes budget asymmetry as a possible explanation for the result:

- **Direct path:** reached the token limit, did not complete the `</think>` block and produced no final answer. Its observable reasoning misclassified CLARO HOGAR, RECARGA NEQUI and TRANSMILENIO as public utilities and converged on **COP 488,450**.
- **Harness path:** also reached the same limit without completing `</think>` or producing a final answer. However, its trace contained exactly one distinct schema-valid plan: `sum`, July 2026, categories `electricity`, `gas` and `water`, status `approved`.
- **Structural recovery:** because $|C(\tau)|=1$, the deterministic executor selected M002, M005 and M008 and returned **COP 325,550**. Zero or multiple valid plans would have produced an explicit failure.

Neither path completed, so the observed difference is not earlier termination. In this run, the contracted trace exposed a single verifiable intermediate object that the harness could execute safely.

Observability converts model behavior into operational information. The termination reason, the presence or absence of `</think>`, structured candidates and contract validation distinguish a complete answer from an incomplete generation or a trace that already contains a usable result. Every harness action can be traced to an observable signal and an explicit rule.

The recorded update reused the already executed 4,096-token direct trace and generated only the matched-budget harness trace. The output reports this explicitly as `Direct trace reused: True`.

> **The harness did not finish the model's reasoning. It converted an observable, verifiable intermediate state into a verifiable system result.**
